# LOCAL 10-K EXTRACTION: Items 1, 1A, and MD&A (Year-by-Year)

**Purpose:** Extract Business (Item 1), Risk Factors (Item 1A), and MD&A (Item 7) from 10-K filings year-by-year on local machine

**Why Year-by-Year?**
- Process 10-Ks incrementally to manage disk space
- Download one year → Extract → Move to next year
- Keep raw and extracted files locally, transfer to SSD when needed

**What This Notebook Does:**
- Extracts Items 1, 1A, and 7 in **one pass** (efficient!)
- Processes years 2025 → 2010 (newest first)
- Monitors disk space
- Semi-automated: confirms before each year
- Supports both rclone download and manual download

**Prerequisites:**
- Google Drive folder: `My Drive/EDGAR_Project/edgar_crawler/datasets/RAW_FILINGS/10-K/`
- Raw 10-K files organized by year (2010/, 2011/, ..., 2025/)
- Sufficient local disk space (~50-100 GB recommended for processing)

---

## SECTION 1: CONFIGURATION

**⚙️ Configure all settings here before running**

In [ ]:
# ============================================================================
# CONFIGURATION - EDIT THESE SETTINGS
# ============================================================================

import os

# --- Local Paths ---
# Where this repository is located on your local machine
REPO_DIR = os.path.expanduser('~/edgar-crawler')  # Change if different

# Where to store raw 10-K files downloaded from Google Drive
LOCAL_RAW_FILINGS_DIR = os.path.join(REPO_DIR, 'datasets', 'RAW_FILINGS', '10-K')

# Where to store extracted JSON files
LOCAL_EXTRACTED_DIR = os.path.join(REPO_DIR, 'datasets', 'EXTRACTED_FILINGS', '10-K')

# --- Google Drive Settings ---
# Path in Google Drive where raw 10-K files are stored
# For rclone, format: "remote_name:path/to/folder"
# Example: "gdrive:EDGAR_Project/edgar_crawler/datasets/RAW_FILINGS/10-K"
GDRIVE_RAW_FILINGS_PATH = "gdrive:EDGAR_Project/edgar_crawler/datasets/RAW_FILINGS/10-K"

# --- Year Range ---
# Will process from YEAR_START down to YEAR_END (newest to oldest)
YEAR_START = 2025  # Start with newest year
YEAR_END = 2010    # End with oldest year

# --- Download Method ---
# "rclone" = Automatically download using rclone (recommended)
# "manual" = You download year folders manually before running
DOWNLOAD_METHOD = "manual"  # Change to "rclone" if you set up rclone

# --- Extraction Settings ---
# Items to extract
ITEMS_TO_EXTRACT = ['1', '1A', '7']  # Business, Risk Factors, MD&A

# Remove financial tables? (recommended for NLP)
REMOVE_TABLES = True

# Number of parallel processes for extraction (adjust based on your CPU)
# Rule of thumb: Use (CPU cores - 1), max 4 for disk I/O efficiency
import multiprocessing
NUM_PROCESSES = min(4, max(1, multiprocessing.cpu_count() - 1))

# --- Disk Space Management ---
# Minimum free disk space (GB) before warning
MIN_FREE_SPACE_GB = 20

# Pause between years for confirmation?
CONFIRM_BEFORE_NEXT_YEAR = True

# ============================================================================

print("✅ Configuration loaded:")
print(f"   Repository: {REPO_DIR}")
print(f"   Year range: {YEAR_START} → {YEAR_END}")
print(f"   Download method: {DOWNLOAD_METHOD}")
print(f"   Items to extract: {', '.join(ITEMS_TO_EXTRACT)}")
print(f"   Parallel processes: {NUM_PROCESSES}")
print(f"   Remove tables: {REMOVE_TABLES}")

---

## SECTION 2: SETUP & DEPENDENCIES

Run once at the start

In [ ]:
# Navigate to repository
import os

if os.path.exists(REPO_DIR):
    os.chdir(REPO_DIR)
    print(f"✅ Working directory: {os.getcwd()}")
else:
    print(f"❌ Repository not found: {REPO_DIR}")
    print(f"   Please update REPO_DIR in the configuration cell above")

In [ ]:
# Install Python dependencies
print("📦 Installing Python dependencies...\n")

!pip install -q beautifulsoup4 lxml requests pandas tqdm click cssutils numpy pyarrow
!pip install -q 'dill<0.3.9' 'multiprocess<0.70.17' pox ppft
!pip install -q --no-deps pathos

print("\n✅ Python dependencies installed")

In [ ]:
# Check if rclone is installed (only if using rclone method)
if DOWNLOAD_METHOD == "rclone":
    import subprocess
    
    try:
        result = subprocess.run(['rclone', 'version'], capture_output=True, text=True)
        if result.returncode == 0:
            print("✅ rclone is installed")
            print(result.stdout.split('\n')[0])
        else:
            print("❌ rclone not found")
    except FileNotFoundError:
        print("❌ rclone not installed")
        print("\n📖 To install rclone:")
        print("   - macOS: brew install rclone")
        print("   - Linux: sudo apt install rclone  OR  curl https://rclone.org/install.sh | sudo bash")
        print("   - Windows: Download from https://rclone.org/downloads/")
        print("\n   Then configure: rclone config")
        print("   (Create a remote named 'gdrive' for Google Drive)")
else:
    print("ℹ️  Download method: manual")
    print("   You'll download year folders manually before extraction")

In [ ]:
# Create necessary directories
import os

os.makedirs(LOCAL_RAW_FILINGS_DIR, exist_ok=True)
os.makedirs(LOCAL_EXTRACTED_DIR, exist_ok=True)

print("✅ Directories created:")
print(f"   Raw filings: {LOCAL_RAW_FILINGS_DIR}")
print(f"   Extracted JSONs: {LOCAL_EXTRACTED_DIR}")

---

## SECTION 3: HELPER FUNCTIONS

Utility functions for disk space, downloading, extraction, etc.

In [ ]:
# Disk space monitoring
import shutil

def check_disk_space(path='.'):
    """Check available disk space in GB"""
    total, used, free = shutil.disk_usage(path)
    free_gb = free // (2**30)  # Convert to GB
    total_gb = total // (2**30)
    used_gb = used // (2**30)
    used_pct = (used / total) * 100
    
    return {
        'total_gb': total_gb,
        'used_gb': used_gb,
        'free_gb': free_gb,
        'used_pct': used_pct
    }

def print_disk_space(path='.'):
    """Print disk space information"""
    space = check_disk_space(path)
    print(f"💾 Disk Space:")
    print(f"   Total: {space['total_gb']:,} GB")
    print(f"   Used: {space['used_gb']:,} GB ({space['used_pct']:.1f}%)")
    print(f"   Free: {space['free_gb']:,} GB")
    
    if space['free_gb'] < MIN_FREE_SPACE_GB:
        print(f"\n⚠️  WARNING: Low disk space! ({space['free_gb']} GB remaining)")
        print(f"   Consider transferring files to SSD before continuing")
        return False
    return True

print("✅ Disk space monitoring functions loaded")

In [ ]:
# Download functions
import subprocess
import os

def download_year_with_rclone(year):
    """Download one year of raw 10-K files using rclone"""
    year_str = str(year)
    remote_path = f"{GDRIVE_RAW_FILINGS_PATH}/{year_str}"
    local_path = os.path.join(LOCAL_RAW_FILINGS_DIR, year_str)
    
    print(f"\n📥 Downloading {year} from Google Drive using rclone...")
    print(f"   From: {remote_path}")
    print(f"   To: {local_path}")
    
    os.makedirs(local_path, exist_ok=True)
    
    # Use rclone copy with progress
    cmd = [
        'rclone', 'copy',
        remote_path,
        local_path,
        '--progress',
        '--transfers', '4',
        '--checkers', '8',
        '--drive-chunk-size', '128M'
    ]
    
    try:
        result = subprocess.run(cmd, check=True)
        print(f"\n✅ Download complete for {year}")
        return True
    except subprocess.CalledProcessError as e:
        print(f"\n❌ Download failed: {e}")
        return False

def check_year_files_exist(year):
    """Check if raw files for a year exist locally"""
    year_path = os.path.join(LOCAL_RAW_FILINGS_DIR, str(year))
    
    if not os.path.exists(year_path):
        return False, 0
    
    # Count raw filing files (txt, htm, html)
    files = [f for f in os.listdir(year_path) 
             if f.endswith(('.txt', '.htm', '.html'))]
    
    return len(files) > 0, len(files)

print("✅ Download functions loaded")

In [ ]:
# Extraction progress tracking
import os
import json

def count_extracted_files(year):
    """Count how many files have been extracted for a given year"""
    year_path = os.path.join(LOCAL_EXTRACTED_DIR, str(year))
    
    if not os.path.exists(year_path):
        return 0
    
    json_files = [f for f in os.listdir(year_path) if f.endswith('.json')]
    return len(json_files)

def get_extraction_stats(year):
    """Get detailed extraction statistics for a year"""
    year_path = os.path.join(LOCAL_EXTRACTED_DIR, str(year))
    
    if not os.path.exists(year_path):
        return {
            'total_files': 0,
            'has_item_1': 0,
            'has_item_1a': 0,
            'has_item_7': 0,
            'has_all_three': 0
        }
    
    json_files = [f for f in os.listdir(year_path) if f.endswith('.json')]
    
    stats = {
        'total_files': len(json_files),
        'has_item_1': 0,
        'has_item_1a': 0,
        'has_item_7': 0,
        'has_all_three': 0
    }
    
    # Sample up to 100 files for quick stats
    sample_size = min(100, len(json_files))
    import random
    sample_files = random.sample(json_files, sample_size) if len(json_files) > sample_size else json_files
    
    for filename in sample_files:
        filepath = os.path.join(year_path, filename)
        try:
            with open(filepath, 'r') as f:
                data = json.load(f)
            
            has_1 = 'item_1' in data and len(data.get('item_1', '')) > 100
            has_1a = 'item_1a' in data and len(data.get('item_1a', '')) > 100
            has_7 = 'item_7' in data and len(data.get('item_7', '')) > 100
            
            if has_1:
                stats['has_item_1'] += 1
            if has_1a:
                stats['has_item_1a'] += 1
            if has_7:
                stats['has_item_7'] += 1
            if has_1 and has_1a and has_7:
                stats['has_all_three'] += 1
        except:
            pass
    
    # Scale up if sampled
    if sample_size < len(json_files):
        scale = len(json_files) / sample_size
        stats['has_item_1'] = int(stats['has_item_1'] * scale)
        stats['has_item_1a'] = int(stats['has_item_1a'] * scale)
        stats['has_item_7'] = int(stats['has_item_7'] * scale)
        stats['has_all_three'] = int(stats['has_all_three'] * scale)
    
    return stats

print("✅ Extraction tracking functions loaded")

---

## SECTION 4: YEAR-BY-YEAR PROCESSING

**Main extraction loop - processes each year from newest to oldest**

In [ ]:
# Initialize processing summary
processing_summary = []

print("="*70)
print(" YEAR-BY-YEAR EXTRACTION")
print("="*70)
print(f"\nProcessing years: {YEAR_START} → {YEAR_END}")
print(f"Items to extract: {', '.join(ITEMS_TO_EXTRACT)}")
print(f"Download method: {DOWNLOAD_METHOD}")
print(f"\n" + "="*70)

In [ ]:
# Process each year
import time
import json
import subprocess

# Generate year list (newest to oldest)
years_to_process = list(range(YEAR_START, YEAR_END - 1, -1))

for idx, year in enumerate(years_to_process):
    print(f"\n\n{'='*70}")
    print(f" PROCESSING YEAR: {year} ({idx + 1}/{len(years_to_process)})")
    print(f"{'='*70}\n")
    
    # Step 1: Check disk space
    print("📊 Step 1: Check disk space")
    if not print_disk_space():
        response = input("\n⚠️  Low disk space! Continue anyway? (yes/no): ")
        if response.lower() != 'yes':
            print("\n⏸️  Processing paused. Transfer files to SSD and resume.")
            break
    
    # Step 2: Check if raw files exist
    print(f"\n📂 Step 2: Check raw files for {year}")
    files_exist, num_raw_files = check_year_files_exist(year)
    
    if files_exist:
        print(f"   ✅ Found {num_raw_files:,} raw 10-K files for {year}")
    else:
        print(f"   ❌ No raw files found for {year}")
        
        if DOWNLOAD_METHOD == "rclone":
            # Download using rclone
            success = download_year_with_rclone(year)
            if not success:
                print(f"\n⚠️  Skipping {year} - download failed")
                processing_summary.append({
                    'year': year,
                    'status': 'failed',
                    'reason': 'download_failed'
                })
                continue
            files_exist, num_raw_files = check_year_files_exist(year)
        else:
            # Manual download
            print(f"\n   📥 Please download {year} folder from Google Drive:")
            print(f"      From: My Drive/EDGAR_Project/edgar_crawler/datasets/RAW_FILINGS/10-K/{year}/")
            print(f"      To: {os.path.join(LOCAL_RAW_FILINGS_DIR, str(year))}/")
            
            response = input(f"\n   Have you downloaded {year} files? (yes/skip): ")
            if response.lower() != 'yes':
                print(f"\n   ⏭️  Skipping {year}")
                processing_summary.append({
                    'year': year,
                    'status': 'skipped',
                    'reason': 'manual_download_not_ready'
                })
                continue
            
            files_exist, num_raw_files = check_year_files_exist(year)
            if not files_exist:
                print(f"   ❌ Still no files found. Skipping {year}")
                processing_summary.append({
                    'year': year,
                    'status': 'skipped',
                    'reason': 'files_not_found'
                })
                continue
    
    # Step 3: Check extraction status
    print(f"\n🔍 Step 3: Check extraction status for {year}")
    num_extracted = count_extracted_files(year)
    print(f"   Already extracted: {num_extracted:,} files")
    print(f"   Raw files: {num_raw_files:,} files")
    
    if num_extracted > 0:
        print(f"   ℹ️  Extraction will skip already-extracted files")
    
    # Step 4: Create extraction config for this year
    print(f"\n⚙️  Step 4: Configure extraction for {year}")
    
    # Create temporary extraction config
    extraction_config = {
        "extract_items": {
            "raw_filings_folder": os.path.join(LOCAL_RAW_FILINGS_DIR, str(year)),
            "extracted_filings_folder": os.path.join(LOCAL_EXTRACTED_DIR, str(year)),
            "filing_types": ["10-K"],
            "items_to_extract": ITEMS_TO_EXTRACT,
            "remove_tables": REMOVE_TABLES,
            "skip_extracted_filings": True,
            "include_signature": False,
            "special_items": {
                "enabled": False
            }
        }
    }
    
    temp_config_path = f'temp_extraction_config_{year}.json'
    with open(temp_config_path, 'w') as f:
        json.dump(extraction_config, f, indent=2)
    
    print(f"   ✅ Config created: {temp_config_path}")
    
    # Step 5: Run extraction
    print(f"\n🚀 Step 5: Extract Items {', '.join(ITEMS_TO_EXTRACT)} for {year}")
    print(f"   Using {NUM_PROCESSES} parallel processes")
    print(f"   This may take a while...\n")
    
    start_time = time.time()
    
    # Use flexible_extractor_fast.py if available, otherwise flexible_extractor.py
    extraction_script = 'flexible_extractor_fast.py' if os.path.exists('flexible_extractor_fast.py') else 'flexible_extractor.py'
    
    cmd = [
        'python', extraction_script,
        '--config', temp_config_path
    ]
    
    if 'fast' in extraction_script:
        cmd.extend(['--processes', str(NUM_PROCESSES)])
    
    try:
        subprocess.run(cmd, check=True)
        elapsed = time.time() - start_time
        
        print(f"\n✅ Extraction complete for {year}!")
        print(f"   Time elapsed: {elapsed/60:.1f} minutes")
        
        # Clean up temp config
        os.remove(temp_config_path)
        
    except subprocess.CalledProcessError as e:
        print(f"\n❌ Extraction failed for {year}: {e}")
        processing_summary.append({
            'year': year,
            'status': 'failed',
            'reason': 'extraction_error'
        })
        continue
    
    # Step 6: Verify results
    print(f"\n📊 Step 6: Verify extraction results for {year}")
    stats = get_extraction_stats(year)
    
    print(f"\n   Extraction Statistics:")
    print(f"   {'─'*50}")
    print(f"   Total extracted files: {stats['total_files']:,}")
    print(f"   With Item 1 (Business): {stats['has_item_1']:,}")
    print(f"   With Item 1A (Risk Factors): {stats['has_item_1a']:,}")
    print(f"   With Item 7 (MD&A): {stats['has_item_7']:,}")
    print(f"   With ALL three items: {stats['has_all_three']:,}")
    
    processing_summary.append({
        'year': year,
        'status': 'completed',
        'num_files': stats['total_files'],
        'elapsed_minutes': elapsed/60,
        'stats': stats
    })
    
    # Step 7: Confirm before next year (if enabled)
    if CONFIRM_BEFORE_NEXT_YEAR and idx < len(years_to_process) - 1:
        next_year = years_to_process[idx + 1]
        print(f"\n{'='*70}")
        print(f" YEAR {year} COMPLETE")
        print(f"{'='*70}")
        
        response = input(f"\n▶️  Continue to {next_year}? (yes/no/pause): ")
        
        if response.lower() == 'no':
            print(f"\n⏹️  Stopping. You can resume later by re-running this cell.")
            break
        elif response.lower() == 'pause':
            print(f"\n⏸️  Paused. Run the next cell to continue.")
            break
    
    print(f"\n✅ Year {year} processing complete!\n")

print(f"\n\n{'='*70}")
print(" PROCESSING SUMMARY")
print(f"{'='*70}\n")

for entry in processing_summary:
    year = entry['year']
    status = entry['status']
    
    if status == 'completed':
        print(f"✅ {year}: {entry['num_files']:,} files in {entry['elapsed_minutes']:.1f} min")
    elif status == 'failed':
        print(f"❌ {year}: Failed - {entry['reason']}")
    elif status == 'skipped':
        print(f"⏭️  {year}: Skipped - {entry['reason']}")

print(f"\n{'='*70}")

---

## SECTION 5: CREATE CONSOLIDATED METADATA

Generate metadata CSV across all processed years

In [ ]:
# Create consolidated metadata CSV
import os
import json
import pandas as pd
from tqdm import tqdm

print("📊 Creating consolidated metadata CSV...\n")

metadata_records = []

# Scan all year folders in extracted directory
for year_folder in sorted(os.listdir(LOCAL_EXTRACTED_DIR)):
    year_path = os.path.join(LOCAL_EXTRACTED_DIR, year_folder)
    
    if not os.path.isdir(year_path):
        continue
    
    try:
        year = int(year_folder)
    except:
        continue
    
    json_files = [f for f in os.listdir(year_path) if f.endswith('.json')]
    
    for filename in tqdm(json_files, desc=f"Year {year}", leave=False):
        filepath = os.path.join(year_path, filename)
        
        try:
            with open(filepath, 'r') as f:
                filing = json.load(f)
            
            metadata_records.append({
                'filename': filename,
                'year': year,
                'cik': filing.get('cik', ''),
                'company': filing.get('company', ''),
                'filing_date': filing.get('filing_date', ''),
                'period_of_report': filing.get('period_of_report', ''),
                'has_item_1': 'item_1' in filing and len(filing.get('item_1', '')) > 100,
                'item_1_length': len(filing.get('item_1', '')),
                'has_item_1a': 'item_1a' in filing and len(filing.get('item_1a', '')) > 100,
                'item_1a_length': len(filing.get('item_1a', '')),
                'has_item_7': 'item_7' in filing and len(filing.get('item_7', '')) > 100,
                'item_7_length': len(filing.get('item_7', '')),
                'json_path': filepath
            })
        except Exception as e:
            print(f"\n⚠️  Error processing {filename}: {e}")

df_meta = pd.DataFrame(metadata_records)
meta_path = os.path.join(REPO_DIR, 'datasets', 'local_extraction_metadata.csv')
df_meta.to_csv(meta_path, index=False)

print(f"\n✅ Metadata CSV created!")
print(f"   Location: {meta_path}")
print(f"   Total records: {len(df_meta):,}")
print(f"   File size: {os.path.getsize(meta_path) / 1024:.1f} KB")

print(f"\n📊 Summary Statistics:")
print(f"   Total filings: {len(df_meta):,}")
print(f"   Years covered: {df_meta['year'].min()} - {df_meta['year'].max()}")
print(f"   Filings with Item 1: {df_meta['has_item_1'].sum():,}")
print(f"   Filings with Item 1A: {df_meta['has_item_1a'].sum():,}")
print(f"   Filings with Item 7: {df_meta['has_item_7'].sum():,}")
print(f"   Filings with ALL three: {(df_meta['has_item_1'] & df_meta['has_item_1a'] & df_meta['has_item_7']).sum():,}")

print(f"\n📏 Average Lengths:")
print(f"   Item 1: {df_meta['item_1_length'].mean():,.0f} characters")
print(f"   Item 1A: {df_meta['item_1a_length'].mean():,.0f} characters")
print(f"   Item 7: {df_meta['item_7_length'].mean():,.0f} characters")

print(f"\n📅 Filings by Year:")
year_counts = df_meta['year'].value_counts().sort_index()
for year, count in year_counts.items():
    print(f"   {year}: {count:,} filings")

In [ ]:
# Optional: Create consolidated Parquet file
create_parquet = input("\n📦 Create consolidated Parquet file? (yes/no): ")

if create_parquet.lower() == 'yes':
    print("\n📦 Creating consolidated Parquet file...\n")
    print("   ⚠️  This may take a while and use significant memory")
    
    full_data = []
    
    for year_folder in sorted(os.listdir(LOCAL_EXTRACTED_DIR)):
        year_path = os.path.join(LOCAL_EXTRACTED_DIR, year_folder)
        
        if not os.path.isdir(year_path):
            continue
        
        try:
            year = int(year_folder)
        except:
            continue
        
        json_files = [f for f in os.listdir(year_path) if f.endswith('.json')]
        
        for filename in tqdm(json_files, desc=f"Year {year}", leave=False):
            filepath = os.path.join(year_path, filename)
            
            try:
                with open(filepath, 'r') as f:
                    filing = json.load(f)
                
                full_data.append({
                    'year': year,
                    'cik': filing.get('cik', ''),
                    'company': filing.get('company', ''),
                    'filing_date': filing.get('filing_date', ''),
                    'period_of_report': filing.get('period_of_report', ''),
                    'item_1_text': filing.get('item_1', ''),
                    'item_1a_text': filing.get('item_1a', ''),
                    'item_7_text': filing.get('item_7', '')
                })
            except Exception as e:
                print(f"\n⚠️  Error processing {filename}: {e}")
    
    df_full = pd.DataFrame(full_data)
    parquet_path = os.path.join(REPO_DIR, 'datasets', 'local_extraction_full.parquet')
    df_full.to_parquet(parquet_path, compression='gzip', index=False)
    
    print(f"\n✅ Parquet file created!")
    print(f"   Location: {parquet_path}")
    print(f"   Records: {len(df_full):,}")
    print(f"   File size: {os.path.getsize(parquet_path) / (1024**2):.1f} MB")
else:
    print("\n⏭️  Skipped Parquet creation")

---

## SECTION 6: DISK SPACE MANAGEMENT

Check disk usage and get transfer instructions

In [ ]:
# Check final disk usage
print("="*70)
print(" DISK SPACE SUMMARY")
print("="*70)

print_disk_space()

# Calculate directory sizes
import subprocess

def get_dir_size(path):
    """Get directory size in GB"""
    total = 0
    try:
        for dirpath, dirnames, filenames in os.walk(path):
            for filename in filenames:
                filepath = os.path.join(dirpath, filename)
                if os.path.exists(filepath):
                    total += os.path.getsize(filepath)
    except Exception as e:
        print(f"Error calculating size: {e}")
    return total / (1024**3)  # Convert to GB

print(f"\n📁 Directory Sizes:")
raw_size = get_dir_size(LOCAL_RAW_FILINGS_DIR)
extracted_size = get_dir_size(LOCAL_EXTRACTED_DIR)

print(f"   Raw 10-K files: {raw_size:.2f} GB")
print(f"   Extracted JSONs: {extracted_size:.2f} GB")
print(f"   Total: {raw_size + extracted_size:.2f} GB")

print(f"\n📊 File Counts:")

# Count raw files
raw_count = 0
for root, dirs, files in os.walk(LOCAL_RAW_FILINGS_DIR):
    raw_count += len([f for f in files if f.endswith(('.txt', '.htm', '.html'))])

# Count extracted files
extracted_count = 0
for root, dirs, files in os.walk(LOCAL_EXTRACTED_DIR):
    extracted_count += len([f for f in files if f.endswith('.json')])

print(f"   Raw 10-K files: {raw_count:,}")
print(f"   Extracted JSONs: {extracted_count:,}")

print(f"\n" + "="*70)

In [ ]:
# Instructions for transferring to SSD
print("="*70)
print(" TRANSFER TO SSD INSTRUCTIONS")
print("="*70)

print("\n📝 When you're ready to transfer files to your SSD:\n")

print("1️⃣  Connect your SSD and note its mount point")
print("   Example: /Volumes/MySSD (Mac) or E:\\ (Windows)\n")

print("2️⃣  Create destination directories on SSD:")
print(f"   mkdir -p /path/to/ssd/EDGAR_RAW_10K")
print(f"   mkdir -p /path/to/ssd/EDGAR_EXTRACTED_10K\n")

print("3️⃣  Copy files to SSD:")
print(f"   # Raw 10-K files")
print(f"   cp -r {LOCAL_RAW_FILINGS_DIR}/* /path/to/ssd/EDGAR_RAW_10K/\n")
print(f"   # Extracted JSON files")
print(f"   cp -r {LOCAL_EXTRACTED_DIR}/* /path/to/ssd/EDGAR_EXTRACTED_10K/\n")

print("4️⃣  Verify transfer completed successfully:")
print(f"   # Count files on SSD")
print(f"   find /path/to/ssd/EDGAR_RAW_10K -type f | wc -l")
print(f"   find /path/to/ssd/EDGAR_EXTRACTED_10K -name '*.json' | wc -l\n")

print("5️⃣  After verifying, you can delete local copies to free space:")
print(f"   rm -rf {LOCAL_RAW_FILINGS_DIR}/*")
print(f"   rm -rf {LOCAL_EXTRACTED_DIR}/*\n")

print("⚠️  IMPORTANT: Always verify files on SSD before deleting local copies!\n")

print("="*70)

---

## 🎉 EXTRACTION COMPLETE!

### Summary of Outputs:

1. **Raw 10-K Files** (by year):
   - Location: `datasets/RAW_FILINGS/10-K/`
   - Structure: Year subfolders (2010/, 2011/, ..., 2025/)
   - Each folder contains raw HTML/TXT files from SEC EDGAR

2. **Extracted JSON Files** (by year):
   - Location: `datasets/EXTRACTED_FILINGS/10-K/`
   - Structure: Year subfolders (2010/, 2011/, ..., 2025/)
   - Each file contains: `item_1` (Business), `item_1a` (Risk Factors), `item_7` (MD&A)

3. **Metadata CSV**:
   - Location: `datasets/local_extraction_metadata.csv`
   - Contains: File info, lengths, flags for all extracted items
   - Use for: Quick filtering, analysis planning

4. **Parquet File** (optional, if created):
   - Location: `datasets/local_extraction_full.parquet`
   - Contains: Full text for Items 1, 1A, and 7
   - Use for: Text analysis, NLP, machine learning

### Next Steps:

- **Analyze extracted text** (sentiment, topics, trends)
- **Compare across years** (business evolution, risk changes)
- **Transfer to SSD** when disk space runs low
- **Research applications**: Risk analysis, disclosure quality, predictive modeling

---

**Questions or Issues?**
- Repository: https://github.com/haowenluo/edgar-crawler
- Documentation: See README.md and AVAILABLE_ITEMS.md

---